In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 23 samples and 11724 columns.
First few rows of the data:


,overall survival time (os in months),death,recurrence free survival time (rfs in months),event,samples,diagnosis,tissue type,study group,treatment,SCYL3,...,USP51,ZNF674,KDM5D,DDX3Y,ZFY,USP9Y,RPS4Y1,NLGN4Y,UTY,EIF1AY
0,64.5667,0,12.30,1,Sondel_RNA_A,melanoma,tumor,A,Hu14.18-IL2,2.540495,...,1.072710,2.435068,11.532765,90.812886,20.731276,28.482075,529.672849,17.286358,19.622720,11.087398
1,31.2333,1,0.76,1,Sondel_RNA_B,melanoma,tumor,B,Hu14.18-IL2,2.259527,...,1.780658,2.116987,0.000000,0.055146,0.022062,0.047871,0.125179,0.019280,0.003478,0.000000
2,9.9000,1,0.00,1,Sondel_RNA_C,melanoma,tumor,B,Never Treated,2.744963,...,2.915865,1.182428,13.912200,52.902504,20.271464,44.052409,189.020923,16.856049,22.095420,8.046334
3,53.0667,0,48.76,0,Sondel_RNA_D,melanoma,tumor,B,Hu14.18-IL2,3.266027,...,0.986413,3.671449,21.166866,29.791873,20.176599,38.073402,347.483597,1.953853,24.797011,5.469122
4,9.4667,1,0.16,1,Sondel_RNA_E,melanoma,tumor,A,Hu14.18-IL2,3.014413,...,0.511402,2.174036,0.003904,0.025756,0.038640,0.046031,0.373527,0.000000,0.109634,0.009069


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for overall survival time (os in months) or death: 0
Number of samples with complete data for both variables: 23
Number of censored samples (event_var == 0.0): 11
Number of events (event_var == 1.0): 12
Censored ratio (among complete): 11 / 23 = 0.478 (47.8%)
Number of samples with incomplete data for recurrence free survival time (rfs in months) or event: 0
Number of samples with complete data for both variables: 23
Number of censored samples (event_var == 0.0): 7
Number of events (event_var == 1.0): 16
Censored ratio (among complete): 7 / 23 = 0.304 (30.4%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE133713/description.json
